In [ ]:
import math
import os
import random
import re
import shutil
from collections.abc import Callable, Sequence
from itertools import combinations
from math import ceil
from multiprocessing import Pool, cpu_count
from typing import Any, Literal

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from ipywidgets import Video
from lightning import LightningDataModule, LightningModule
from PIL import Image
from skimage.metrics import structural_similarity
from sklearn.model_selection import GroupShuffleSplit
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, ConfusionMatrix, F1Score, MaxMetric, MeanMetric
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_numpy_images,
    show_pytorch_images,
)

data_dir = root / "data"

# Fix FETAL_PLANES csv

## Assign Subset value

### Load FETAL_PLANE csv

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data.csv")
fetal_planes = fetal_planes[["Image_name", "Patient_num", "Plane", "Brain_plane", "Train"]]
fetal_planes = fetal_planes.reset_index(drop=True)
fetal_planes

### Assign Subset value (the same for all experiments)

In [ ]:
train_fetal_planes = fetal_planes[fetal_planes["Train"] == 1].reset_index(drop=True)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=5724)
split = splitter.split(list(range(len(train_fetal_planes))), groups=train_fetal_planes["Patient_num"])
train_idx, val_idx = next(split)
val_idx = set(val_idx)

fetal_planes["Subset"] = "test"

for idx, image_name in tqdm(enumerate(train_fetal_planes["Image_name"]), total=len(train_fetal_planes)):
    if idx in val_idx:
        subset = "val"
    else:
        subset = "train"
    fetal_planes.loc[fetal_planes["Image_name"] == image_name, "Subset"] = subset

fetal_planes

### Statistics

In [ ]:
fetal_planes["Plane"].value_counts()

In [ ]:
fetal_planes["Brain_plane"].value_counts()

In [ ]:
fetal_planes["Subset"].value_counts()

In [ ]:
fetal_planes[fetal_planes["Plane"] == "Fetal brain"]["Subset"].value_counts()

In [ ]:
def get_subset_brain_planes_df(df, subset):
    if subset != "all":
        df = df[df["Subset"] == subset]
    df = df["Brain_plane"].value_counts().reset_index()
    df.columns = ["Brain_plane", "Count"]
    df.loc[len(df)] = {"Brain_plane": "Total", "Count": df["Count"].sum()}
    df["Subset"] = subset
    return df


def get_brain_planes_count(df):
    all_df = get_subset_brain_planes_df(df, "all")
    train_df = get_subset_brain_planes_df(df, "train")
    val_df = get_subset_brain_planes_df(df, "val")
    test_df = get_subset_brain_planes_df(df, "test")
    df = pd.concat([all_df, train_df, val_df, test_df], ignore_index=True)
    df = df.pivot_table(index="Brain_plane", columns="Subset", values="Count", fill_value=0)
    df = df.reindex(columns=["train", "val", "test", "all"])
    df = df.reindex(["Trans-ventricular", "Trans-thalamic", "Trans-cerebellum", "Other", "Not A Brain", "Total"])
    df = df.astype(int)
    return df


get_brain_planes_count(fetal_planes)

### Save csv

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes

## Remove duplicates

### Load FETAL_PLANE csv

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes

### Find duplicates

In [ ]:
def read_fetal_planes_image(image_name):
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    return read_grayscale_image(image_path)


def read_grayscale_image(image_path):
    image = Image.open(image_path).convert("L")
    image_array = np.asarray(image)
    return image_array


def calculate_ssim(image1, image2):
    if image1.shape != image2.shape:
        return 0.0

    ssim = structural_similarity(image1, image2, data_range=255)
    if ssim > 0.95:
        return ssim
    elif ssim > 0.5:
        image1_crop = image1[:, : int(image1.shape[1] * 0.75)]
        image2_crop = image2[:, : int(image1_crop.shape[1])]
        return structural_similarity(image1_crop, image2_crop, data_range=255)
    else:
        return ssim


def process_patient_data(patient_num):
    """
    The worker function to be run in a separate process.
    It processes all images for a single patient and returns the results.
    """
    # These lists will store the results for this patient only
    local_duplicates = []

    patient_df = fetal_planes[fetal_planes["Patient_num"] == patient_num]
    indexes = patient_df.index.to_list()

    # Read all images for the current patient
    images = {}
    for index in indexes:
        image = read_fetal_planes_image(patient_df.loc[index, "Image_name"])
        images[index] = image

    # Pairwise comparison of all images for the patient
    duplicate_indexes: set[int] = set()
    indexes_pairs = list(combinations(indexes, 2))
    for index1, index2 in indexes_pairs:
        # Skip images already linked to an earlier duplicate for this patient
        if index2 in duplicate_indexes:
            continue

        image1 = images[index1]
        image2 = images[index2]
        ssim = calculate_ssim(image1, image2)

        if ssim > 0.95:
            local_duplicates.append((index1, index2, ssim))
            duplicate_indexes.add(index2)

    return local_duplicates


fetal_planes["Duplicate"] = None

num_processes = cpu_count()
duplicates = []

print(f"Starting parallel processing with {num_processes} cores...")
with Pool(num_processes) as pool:
    patients = fetal_planes["Patient_num"].unique().tolist()
    for local_duplicates in tqdm(pool.imap(process_patient_data, patients), total=len(patients), desc="Patients"):
        duplicates.extend(local_duplicates)

for base_index, duplicate_index, _ in duplicates:
    fetal_planes.loc[duplicate_index, "Duplicate"] = int(base_index)

print("\nProcessing complete.")
print(f"Found {len(duplicates)} similar images.")  # 1072

### Group duplicates

In [ ]:
images = []

root_duplicates = []
rows_visited = set()
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    if index in rows_visited:
        continue

    indexes = [index]
    duplicate_indexes = []
    while len(indexes) > 0:
        current_index = indexes.pop(0)
        rows_visited.add(current_index)
        duplicate_indexes.append(current_index)
        indexes.extend(fetal_planes[fetal_planes["Duplicate"] == current_index].index.to_list())

    if len(duplicate_indexes) > 1:
        root_duplicates.append(duplicate_indexes)

print(len(root_duplicates))

### Display duplicates per group

In [ ]:
images = []

for root_duplicate in tqdm([root_duplicates[i] for i in [0, 60, 500, 600]]):
    base_image = None
    for index in root_duplicate:
        image = read_fetal_planes_image(fetal_planes.Image_name[index])

        if base_image is None:
            base_image = image
            label = f"{index}"
        else:
            ssim = calculate_ssim(base_image, image)
            label = f"{index}: {ssim:.3f}"

        images.append((image, label))

        if len(images) >= 64:
            break
    if len(images) >= 64:
        break

print(len(images))
if images:
    show_numpy_images(images[:36])

### Statistics

In [ ]:
get_brain_planes_count(fetal_planes)

In [ ]:
get_brain_planes_count(fetal_planes[fetal_planes["Duplicate"].isna()])

### Save csv

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes

## Remove invalid data

### Load FETAL_PLANE csv

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes["Valid"] = 1
fetal_planes["New_brain_plane"] = None
fetal_planes

### Remove brain images that are not showing brain

In [ ]:
images = []

remove_images = [
    "Patient00765_Plane3_5_of_6",
    "Patient00765_Plane3_6_of_6",
    "Patient00850_Plane3_4_of_4",
    "Patient01028_Plane3_3_of_3",
    # Other brain images
    "Patient00684_Plane3_2_of_3",
    "Patient00684_Plane3_3_of_3",
    "Patient00721_Plane3_2_of_3",
    "Patient01013_Plane3_4_of_4",
    "Patient01272_Plane3_3_of_3",
    "Patient01304_Plane3_4_of_6",
    "Patient01646_Plane3_3_of_3",
    "Patient00803_Plane3_1_of_5",
    # Axial brain images
    "Patient00914_Plane3_3_of_4",
    "Patient01280_Plane3_3_of_5",
    "Patient01635_Plane3_1_of_4",
]
for image_name in remove_images:
    for index, row in fetal_planes[fetal_planes.Image_name == image_name].iterrows():
        fetal_planes.loc[index, "Valid"] = 0
        image = read_fetal_planes_image(image_name)
        images.append((image, f"{index}: {image_name}"))
        print(f"{index:<6}: {image_name} \t{row['Plane']} \t{row['Brain_plane']}")

if images:
    show_numpy_images(images[:36])

### Remove Not a brain images that are showing brain image

In [ ]:
images = []

# Brain - TT, TC, TV
# Mid-sagittal - head profil
# Axial - top with nose

remove_images = [
    # ("Patient00914_Plane3_3_of_4", "Axial"),
    # ("Patient01280_Plane3_3_of_5", "Axial"),
    # ("Patient01635_Plane3_1_of_4", "Axial"),
    # ("Patient01408_Plane1_1_of_15", "Axial"),
    # ("Patient01721_Plane1_1_of_1", "Axial"),
    # ("Patient01736_Plane1_1_of_1", "Axial"),
    # ("Patient01680_Plane1_1_of_1", "Axial"),
    # ("Patient00168_Plane1_10_of_23", "Axial"),
    # ("Patient01003_Plane1_1_of_1", "Axial"),
    # ("Patient01286_Plane1_1_of_1", "Axial"),
    # ("Patient01327_Plane1_8_of_21", "Axial"),
    # ("Patient01719_Plane1_1_of_1", "Axial"),
    # ("Patient01486_Plane1_13_of_26", "Axial"),
    # ("Patient01790_Plane1_1_of_1", "Axial"),
    # ("Patient01179_Plane1_1_of_1", "Axial"),
    # ("Patient00897_Plane1_1_of_1", "Axial"),
    # ("Patient01617_Plane1_1_of_1", "Axial"),
    # ("Patient01179_Plane1_1_of_1", "Axial"),
    # ("Patient00799_Plane1_21_of_28", "Axial"),
    # ("Patient00897_Plane1_1_of_1", "Axial"),
    # ("Patient01617_Plane1_1_of_1", "Axial"),
    # ("Patient01583_Plane1_1_of_1", "Axial"),
    # ("Patient01347_Plane1_19_of_24", "Axial"),
    # ("Patient00906_Plane1_7_of_27", "Axial"),
    # ("Patient01035_Plane1_1_of_1", "Axial"),
    # ("Patient00779_Plane1_1_of_1", "Axial"),
    # ("Patient00791_Plane1_1_of_25", "Axial"),
    # ("Patient00944_Plane1_1_of_1", "Axial"),
    # ("Patient00830_Plane1_11_of_49", "Axial"),
    # ("Patient01162_Plane1_1_of_1", "Axial"),
    # ("Patient01185_Plane1_9_of_15", "Axial"),
    # ("Patient00890_Plane1_8_of_19", "Axial"),
    # ("Patient00791_Plane1_14_of_25", "Axial"),
    # ("Patient01631_Plane1_2_of_16", "Axial"),
    # ("Patient01373_Plane1_7_of_9", "Axial"),
    # ("Patient00720_Plane1_22_of_27", "Axial"),
    # ("Patient00720_Plane1_7_of_27", "Axial"),
    # ("Patient01365_Plane1_10_of_17", "Axial"),
    # ("Patient00216_Plane1_1_of_1", "Axial"),
    # ("Patient00720_Plane1_27_of_27", "Axial"),
    # ("Patient00834_Plane1_12_of_34", "Axial"),
    # ("Patient01531_Plane1_1_of_1", "Axial"),
    # ("Patient01513_Plane1_14_of_14", "Axial"),
    # ("Patient01517_Plane1_1_of_20", "Axial"),
    # ("Patient01573_Plane1_4_of_8", "Axial"),
    # ("Patient01481_Plane1_33_of_35", "Axial"),
    # ("Patient01566_Plane1_20_of_29", "Axial"),
    # ("Patient00813_Plane1_2_of_10", "Brain"), # +
    # ("Patient00813_Plane1_8_of_10", "Brain"), # +
    # ("Patient00854_Plane1_2_of_32", "Brain"),
    # ("Patient00854_Plane1_12_of_32", "Brain"),
    # ("Patient00855_Plane1_32_of_39", "Brain"),
    # ("Patient00873_Plane1_14_of_23", "Brain"), # +
    # ("Patient00875_Plane1_27_of_37", "Brain"),
    # ("Patient01181_Plane5_3_of_3", "Brain"), # +
    # ("Patient01188_Plane1_1_of_2", "Brain"), # +
    # ("Patient01296_Plane1_1_of_1", "Brain"), # +
    # ("Patient01397_Plane1_3_of_9", "Brain"), # +
    # ("Patient00890_Plane1_5_of_19", "Brain"), # +
    # ("Patient01240_Plane1_1_of_1", "Brain"), # -
    # ("Patient01240_Plane1_1_of_1", "Brain"), # -
    # ("Patient01484_Plane1_12_of_16", "Brain"), # -
    # ("Patient01391_Plane1_1_of_26", "Brain"), # +
    # ("Patient01535_Plane1_5_of_21", "Brain"),
    # ("Patient01535_Plane1_18_of_21", "Brain"),
    # ("Patient01169_Plane4_4_of_4", "Brain"),
    # ("Patient01653_Plane4_1_of_6", "Brain"),
    # ("Patient00764_Plane1_1_of_1", "Brain"),
    # ("Patient00790_Plane1_19_of_47", "Brain"),
    # ("Patient01212_Plane1_1_of_1", "Brain"),
    # ("Patient01198_Plane1_7_of_9", "Brain"),
    # ("Patient00799_Plane1_21_of_28", "Other"),
    # ("Patient01149_Plane1_1_of_1", "Other"),
    # ("Patient00961_Plane1_1_of_1", "Other"),
    # ("Patient01036_Plane1_1_of_1", "Other"),
    # ("Patient00789_Plane1_11_of_64", "Other"),
    # ("Patient00908_Plane1_3_of_4", "Other"),
    # ("Patient01547_Plane1_1_of_18", "Other"),
    # ("Patient00722_Plane1_1_of_1", "Other"),
    # ("Patient01367_Plane1_1_of_8", "Other"),
    # ("Patient01287_Plane1_6_of_10", "Other"),
    # ("Patient01572_Plane1_5_of_9", "Other"),
    # ("Patient01473_Plane1_28_of_30", "Other"),
    # ("Patient01513_Plane1_3_of_14", "Other"),
    # ("Patient00889_Plane1_7_of_12", "Other"),
    # ("Patient00790_Plane1_17_of_47", "Mid-sagittal"),
    # ("Patient00684_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01221_Plane1_6_of_9", "Mid-sagittal"),
    # ("Patient01698_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01287_Plane1_5_of_10", "Mid-sagittal"),
    # ("Patient01547_Plane1_13_of_18", "Mid-sagittal"),
    # ("Patient01500_Plane1_4_of_9", "Mid-sagittal"),
    # ("Patient01445_Plane1_1_of_11", "Mid-sagittal"),
    # ("Patient00799_Plane1_18_of_28", "Mid-sagittal"),
    # ("Patient00790_Plane1_43_of_47", "Mid-sagittal"),
    # ("Patient00799_Plane1_8_of_28", "Mid-sagittal"),
    # ("Patient00799_Plane1_17_of_28", "Mid-sagittal"),
    # ("Patient00830_Plane1_18_of_49", "Mid-sagittal"),
    # ("Patient00875_Plane1_37_of_37", "Mid-sagittal"),
    # ("Patient00893_Plane1_2_of_3", "Mid-sagittal"),
    # ("Patient00799_Plane1_18_of_28", "Mid-sagittal"),
    # ("Patient00790_Plane1_43_of_47", "Mid-sagittal"),
    # ("Patient00799_Plane1_8_of_28", "Mid-sagittal"),
    # ("Patient00799_Plane1_17_of_28", "Mid-sagittal"),
    # ("Patient00830_Plane1_18_of_49", "Mid-sagittal"),
    # ("Patient00875_Plane1_37_of_37", "Mid-sagittal"),
    # ("Patient00893_Plane1_2_of_3", "Mid-sagittal"),
    # ("Patient01683_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01654_Plane1_6_of_12", "Mid-sagittal"),
    # ("Patient01545_Plane1_3_of_13", "Mid-sagittal"),
    # ("Patient01548_Plane1_1_of_7", "Mid-sagittal"),
    # ("Patient00830_Plane1_2_of_49", "Mid-sagittal"),
    # ("Patient01513_Plane1_6_of_14", "Mid-sagittal"),
    # ("Patient01513_Plane1_8_of_14", "Mid-sagittal"),
    # ("Patient00792_Plane1_29_of_71", "Mid-sagittal"),
    # ("Patient00875_Plane1_2_of_37", "Mid-sagittal"),
    # ("Patient00889_Plane1_4_of_12", "Mid-sagittal"),
    # ("Patient01347_Plane1_13_of_24", "Mid-sagittal"),
    # ("Patient01448_Plane1_6_of_13", "Mid-sagittal"),
    # ("Patient01455_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01473_Plane1_9_of_30", "Mid-sagittal"),
    # ("Patient01484_Plane1_8_of_16", "Mid-sagittal"),
    # ("Patient01517_Plane1_5_of_20", "Mid-sagittal"),
    # ("Patient01248_Plane1_5_of_15", "Mid-sagittal"),
    # ("Patient01347_Plane1_7_of_24", "Mid-sagittal"),
    # ("Patient01422_Plane1_2_of_19", "Mid-sagittal"),
    # ("Patient00790_Plane1_44_of_47", "Mid-sagittal"),
    # ("Patient00742_Plane1_15_of_20", "Mid-sagittal"),
    # ("Patient00793_Plane1_10_of_48", "Mid-sagittal"),
    # ("Patient00793_Plane1_33_of_48", "Mid-sagittal"),
    # ("Patient00835_Plane1_23_of_37", "Mid-sagittal"),
    # ("Patient01444_Plane1_25_of_28", "Mid-sagittal"),
    # ("Patient01454_Plane1_4_of_6", "Mid-sagittal"),
    # ("Patient01549_Plane1_2_of_4", "Mid-sagittal"),
    # ("Patient01549_Plane1_4_of_4", "Mid-sagittal"),
    # ("Patient01551_Plane1_8_of_22", "Mid-sagittal"),
    # ("Patient01570_Plane1_9_of_16", "Mid-sagittal"),
    # ("Patient01606_Plane1_5_of_18", "Mid-sagittal"),
    # ("Patient00890_Plane1_4_of_19", "Mid-sagittal"),
    # ("Patient01308_Plane1_8_of_9", "Mid-sagittal"),
    # ("Patient01332_Plane1_8_of_12", "Mid-sagittal"),
    # ("Patient01612_Plane1_7_of_21", "Mid-sagittal"),
    # ("Patient01513_Plane1_10_of_14", "Mid-sagittal"),
    # ("Patient00792_Plane1_16_of_71", "Mid-sagittal"),
    # ("Patient01365_Plane1_13_of_17", "Mid-sagittal"),
    # ("Patient01395_Plane1_6_of_8", "Mid-sagittal"),
    # ("Patient00168_Plane1_4_of_23", "Mid-sagittal"),
    # ("Patient00768_Plane1_22_of_26", "Mid-sagittal"),
    # ("Patient01738_Plane1_2_of_2", "Mid-sagittal"),
    # ("Patient00768_Plane1_23_of_26", "Mid-sagittal"),
    # ("Patient00791_Plane1_20_of_25", "Mid-sagittal"),
    # ("Patient00792_Plane1_35_of_71", "Mid-sagittal"),
    # ("Patient00909_Plane1_13_of_15", "Mid-sagittal"),
    # ("Patient01185_Plane1_12_of_15", "Mid-sagittal"),
    # ("Patient01120_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01481_Plane1_17_of_35", "Mid-sagittal"),
    # ("Patient00770_Plane1_1_of_2", "Mid-sagittal"),
    # ("Patient01444_Plane1_11_of_28", "Mid-sagittal"),
    # ("Patient01446_Plane1_17_of_29", "Mid-sagittal"),
    # ("Patient01450_Plane1_10_of_14", "Mid-sagittal"),
    # ("Patient00791_Plane1_2_of_25", "Mid-sagittal"),
    # ("Patient00810_Plane1_3_of_11", "Mid-sagittal"),
    # ("Patient00834_Plane1_13_of_34", "Mid-sagittal"),
    # ("Patient00835_Plane1_8_of_37", "Mid-sagittal"),
    # ("Patient01348_Plane1_5_of_8", "Mid-sagittal"),
    # ("Patient01548_Plane1_7_of_7", "Mid-sagittal"),
    # ("Patient00168_Plane1_7_of_23", "Mid-sagittal"),
    # ("Patient00168_Plane1_20_of_23", "Mid-sagittal"),
    # ("Patient00188_Plane1_2_of_3", "Mid-sagittal"),
    # ("Patient00188_Plane1_3_of_3", "Mid-sagittal"),
    # ("Patient00720_Plane1_10_of_27", "Mid-sagittal"),
    # ("Patient00739_Plane1_6_of_8", "Mid-sagittal"),
    # ("Patient00768_Plane1_1_of_26", "Mid-sagittal"),
    # ("Patient00768_Plane1_3_of_26", "Mid-sagittal"),
    # ("Patient00769_Plane1_6_of_17", "Mid-sagittal"),
    # ("Patient00771_Plane1_26_of_40", "Mid-sagittal"),
    # ("Patient00789_Plane1_8_of_64", "Mid-sagittal"),
    # ("Patient00791_Plane1_9_of_25", "Mid-sagittal"),
    # ("Patient00793_Plane1_32_of_48", "Mid-sagittal"),
    # ("Patient00795_Plane1_27_of_41", "Mid-sagittal"),
    # ("Patient00795_Plane1_35_of_41", "Mid-sagittal"),
    # ("Patient00795_Plane1_37_of_41", "Mid-sagittal"),
    # ("Patient00809_Plane1_11_of_14", "Mid-sagittal"),
    # ("Patient00813_Plane1_10_of_10", "Mid-sagittal"),
    # ("Patient00831_Plane1_2_of_38", "Mid-sagittal"),
    # ("Patient00831_Plane1_13_of_38", "Mid-sagittal"),
    # ("Patient00831_Plane1_26_of_38", "Mid-sagittal"),
    # ("Patient00854_Plane1_24_of_32", "Mid-sagittal"),
    # ("Patient00854_Plane1_25_of_32", "Mid-sagittal"),
    # ("Patient00854_Plane1_28_of_32", "Mid-sagittal"),
    # ("Patient00855_Plane1_7_of_39", "Mid-sagittal"),
    # ("Patient00855_Plane1_11_of_39", "Mid-sagittal"),
    # ("Patient00855_Plane1_16_of_39", "Mid-sagittal"),
    # ("Patient00869_Plane1_2_of_5", "Mid-sagittal"),
    # ("Patient00870_Plane1_2_of_8", "Mid-sagittal"),
    # ("Patient00871_Plane1_4_of_11", "Mid-sagittal"),
    # ("Patient00871_Plane1_5_of_11", "Mid-sagittal"),
    # ("Patient00873_Plane1_11_of_23", "Mid-sagittal"),
    # ("Patient00873_Plane1_15_of_23", "Mid-sagittal"),
    # ("Patient00909_Plane1_14_of_15", "Mid-sagittal"),
    # ("Patient01032_Plane1_1_of_4", "Mid-sagittal"),
    # ("Patient01032_Plane1_2_of_4", "Mid-sagittal"),
    # ("Patient01188_Plane1_2_of_2", "Mid-sagittal"),
    # ("Patient01198_Plane1_3_of_9", "Mid-sagittal"),
    # ("Patient01198_Plane1_4_of_9", "Mid-sagittal"),
    # ("Patient01198_Plane1_8_of_9", "Mid-sagittal"),
    # ("Patient01198_Plane1_9_of_9", "Mid-sagittal"),
    # ("Patient01199_Plane1_2_of_11", "Mid-sagittal"),
    # ("Patient01215_Plane1_6_of_24", "Mid-sagittal"),
    # ("Patient01215_Plane1_8_of_24", "Mid-sagittal"),
    # ("Patient01217_Plane1_1_of_5", "Mid-sagittal"),
    # ("Patient01217_Plane1_3_of_5", "Mid-sagittal"),
    # ("Patient01217_Plane1_4_of_5", "Mid-sagittal"),
    # ("Patient01219_Plane1_1_of_2", "Mid-sagittal"),
    # ("Patient01259_Plane1_2_of_7", "Mid-sagittal"),
    # ("Patient01272_Plane1_7_of_14", "Mid-sagittal"),
    # ("Patient01288_Plane1_1_of_4", "Mid-sagittal"),
    # ("Patient01307_Plane1_2_of_25", "Mid-sagittal"),
    # ("Patient01331_Plane1_1_of_4", "Mid-sagittal"),
    # ("Patient01364_Plane1_2_of_6", "Mid-sagittal"),
    # ("Patient01364_Plane1_3_of_6", "Mid-sagittal"),
    # ("Patient01365_Plane1_16_of_17", "Mid-sagittal"),
    # ("Patient01367_Plane1_4_of_8", "Mid-sagittal"),
    # ("Patient01374_Plane1_1_of_12", "Mid-sagittal"),
    # ("Patient01374_Plane1_8_of_12", "Mid-sagittal"),
    # ("Patient01391_Plane1_23_of_26", "Mid-sagittal"),
    # ("Patient01398_Plane1_1_of_10", "Mid-sagittal"),
    # ("Patient01398_Plane1_7_of_10", "Mid-sagittal"),
    # ("Patient01398_Plane1_8_of_10", "Mid-sagittal"),
    # ("Patient01402_Plane1_6_of_8", "Mid-sagittal"),
    # ("Patient01408_Plane1_5_of_15", "Mid-sagittal"),
    # ("Patient01408_Plane1_6_of_15", "Mid-sagittal"),
    # ("Patient01409_Plane1_6_of_27", "Mid-sagittal"),
    # ("Patient01409_Plane1_7_of_27", "Mid-sagittal"),
    # ("Patient01409_Plane1_19_of_27", "Mid-sagittal"),
    # ("Patient01442_Plane1_21_of_22", "Mid-sagittal"),
    # ("Patient01446_Plane1_16_of_29", "Mid-sagittal"),
    # ("Patient01448_Plane1_2_of_13", "Mid-sagittal"),
    # ("Patient01448_Plane1_3_of_13", "Mid-sagittal"),
    # ("Patient01448_Plane1_9_of_13", "Mid-sagittal"),
    # ("Patient01452_Plane1_3_of_9", "Mid-sagittal"),
    # ("Patient01452_Plane1_4_of_9", "Mid-sagittal"),
    # ("Patient01454_Plane1_2_of_6", "Mid-sagittal"),
    # ("Patient01469_Plane1_5_of_7", "Mid-sagittal"),
    # ("Patient01473_Plane1_15_of_30", "Mid-sagittal"),
    # ("Patient01481_Plane1_6_of_35", "Mid-sagittal"),
    # ("Patient01485_Plane1_7_of_18", "Mid-sagittal"),
    # ("Patient01487_Plane1_3_of_17", "Mid-sagittal"),
    # ("Patient01487_Plane1_17_of_17", "Mid-sagittal"),
    # ("Patient01498_Plane1_1_of_3", "Mid-sagittal"),
    # ("Patient01505_Plane1_12_of_24", "Mid-sagittal"),
    # ("Patient01505_Plane1_24_of_24", "Mid-sagittal"),
    # ("Patient01510_Plane1_2_of_12", "Mid-sagittal"),
    # ("Patient01517_Plane1_11_of_20", "Mid-sagittal"),
    # ("Patient01517_Plane1_20_of_20", "Mid-sagittal"),
    # ("Patient01535_Plane1_12_of_21", "Mid-sagittal"),
    # ("Patient01539_Plane1_14_of_25", "Mid-sagittal"),
    # ("Patient01545_Plane1_2_of_13", "Mid-sagittal"),
    # ("Patient01547_Plane1_5_of_18", "Mid-sagittal"),
    # ("Patient01551_Plane1_3_of_22", "Mid-sagittal"),
    # ("Patient01566_Plane1_15_of_29", "Mid-sagittal"),
    # ("Patient01570_Plane1_15_of_16", "Mid-sagittal"),
    # ("Patient01571_Plane1_1_of_1", "Mid-sagittal"),
    # ("Patient01613_Plane1_10_of_15", "Mid-sagittal"),
    # ("Patient01655_Plane1_7_of_12", "Mid-sagittal"),
    # ("Patient01655_Plane1_8_of_12", "Mid-sagittal"),
    # ("Patient01659_Plane1_4_of_6", "Mid-sagittal"),
    # ("Patient01671_Plane1_3_of_3", "Mid-sagittal"),
    # ("Patient01185_Plane1_11_of_15", "Mid-sagittal"),
    # ("Patient01249_Plane1_1_of_7", "Mid-sagittal"),
    # ("Patient01327_Plane1_2_of_21", "Mid-sagittal"),
    # ("Patient01287_Plane1_7_of_10", "Mid-sagittal"),
    # ("Patient01485_Plane1_2_of_18", "Mid-sagittal"),
    # ("Patient01218_Plane1_6_of_11", "Mid-sagittal"),
    # ("Patient00853_Plane1_1_of_2", "Mid-sagittal"),
    # ("Patient01572_Plane1_6_of_9", "Mid-sagittal"),
    # ("Patient01485_Plane1_11_of_18", "Mid-sagittal"),
    # ("Patient01438_Plane1_14_of_15", "Mid-sagittal"), # -1
]
for image_name, new_brain_plane in remove_images:
    for index, row in fetal_planes[fetal_planes.Image_name == image_name].iterrows():
        fetal_planes.loc[index, "New_brain_plane"] = new_brain_plane
        image = read_fetal_planes_image(image_name)
        images.append((image, f"{index}: {new_brain_plane}"))
        # print(f"\"{image_name}\"")
        # print(f"{index:<6}: {image_name} \t{row['Plane']}")
        # print(f"{index}, ", end="")


print(len(images))
if images:
    show_numpy_images(images[:36])

In [ ]:
images = []

remove_images = [5899]
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    image = read_fetal_planes_image(image_name)
    images.append((image, f"{index}: {image_name}"))
    # print(f"\"{image_name}\"")
    print(f"{index:<6}: {image_name} \t{fetal_planes.Plane[index]}")
    # print(f"    (\"{image_name}\", \"Other\"),")

print()
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    print(f'"{image_name}"')

print()
for index in remove_images:
    image_name = fetal_planes.Image_name[index]
    print(f'    ("{image_name}", "Other"),')

print(len(images))
if images:
    show_numpy_images(images[:36])

### Statistics

In [ ]:
get_brain_planes_count(fetal_planes[fetal_planes["Duplicate"].isna()])

In [ ]:
df = fetal_planes[fetal_planes["Duplicate"].isna() & (fetal_planes["Valid"] == 1)].copy()
for index, row in df.iterrows():
    if isinstance(row["New_brain_plane"], str):
        df.loc[index, "Brain_plane"] = "Other"

get_brain_planes_count(df)

In [ ]:
# Same sanity check as the cell above (kept for re-running after manual label edits).
df = fetal_planes[fetal_planes["Duplicate"].isna() & (fetal_planes["Valid"] == 1)].copy()
for index, row in df.iterrows():
    if isinstance(row["New_brain_plane"], str):
        df.loc[index, "Brain_plane"] = "Other"

get_brain_planes_count(df)

### Save csv

In [ ]:
fetal_planes.to_csv(data_dir / "FETAL_PLANES" / "data_fix.csv", index=False)
fetal_planes